### 说明
AgentChat属于实验阶段不稳定，当前版本代码报错，了解即可，可以换版本试试，下面的官网说明地址
https://learn.microsoft.com/en-us/semantic-kernel/support/archive/agent-chat?pivots=programming-language-python

AUTOGEN相对成熟，可以直接看08-autogen.ipynb,代码功能是一样的

In [6]:
import os

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import AuthorRole, ChatMessageContent
from semantic_kernel.functions import KernelFunctionFromPrompt

from dotenv import load_dotenv

load_dotenv()

def _create_kernel_with_chat_completion() -> Kernel:
    """
    创建并配置Semantic Kernel实例，连接到LLM服务
    
    Returns:
        Kernel: 配置好的Kernel实例
    """
    kernel = Kernel()

     # 创建异步OpenAI客户端，使用GitHub提供的AI模型服务
    client = AsyncOpenAI(
        api_key=os.environ["GITHUB_TOKEN"], 
        base_url="https://models.inference.ai.azure.com/",
    )

    # 将聊天完成服务添加到Kernel
    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4o-mini",
            async_client=client,
        )
    )

    return kernel

async def main():
    """演示多代理对话系统，包含前台旅行代理和礼宾员"""

    # 定义礼宾Agent
    # 礼宾的提示词（中文版本）
    # 你是一位酒店礼宾，对为旅行者提供最本地化和真实的体验有独到见解。
    # 目标是判断前台旅行代理是否为旅行者推荐了最佳非旅游景点体验。
    # 如果是，请声明已批准。
    # 如果不是，请提供如何改进推荐的见解，但不要使用具体示例。
    REVIEWER_NAME = "Concierge"
    REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """
    agent_reviewer = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion(),
        name=REVIEWER_NAME,
        instructions=REVIEWER_INSTRUCTIONS,
    )

    # 定义前台旅行代理Agent
    # 前台的提示词（中文版本）
    # 你是一位拥有十年经验的前台旅行代理，以简洁著称，因为你需要处理众多客户。
    # 目标是为旅行者提供最佳活动和地点推荐。
    # 每次回复仅提供一个推荐。
    # 你专注于手头的目标。
    # 不要浪费时间闲聊。
    # 考虑改进建议时的反馈。
    FRONTDESK_NAME = "FrontDesk"
    FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """
    agent_writer = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion(),
        name=FRONTDESK_NAME,
        instructions=FRONTDESK_INSTRUCTIONS,
    )

    # 定义终止函数 - 判断对话是否完成
    # 提示词中文：
    # 确定推荐流程是否完成。
    # 当礼宾批准前台旅行代理的任何推荐时，流程即完成。
    # 寻找"批准"、"此推荐已批准"或任何表明礼宾对建议满意的明确表述。
    # 如果礼宾在最新回复中已批准，回复：是
    # 否则，回复：否
    # 历史记录：
    # {{$history}}
    termination_function = KernelFunctionFromPrompt(
        function_name="termination",
        prompt="""
        Determine if the recommendation process is complete.
        
        The process is complete when the Concierge provides approval for any recommendation made by the Front Desk.
        Look for phrases like "approved", "this recommendation is approved", or any clear indication that the Concierge is satisfied with the suggestion.
        
        If the Concierge has given approval in their most recent response, respond with: yes
        Otherwise, respond with: no
        
        History:
        {{$history}}
        """,
    )

    # 定义选择函数 - 选择下一个发言者
    # 提示词中文：
    # 根据最近发言者，确定对话中下一位参与者。
    # 仅说出下一位参与者的名称。
    # 没有参与者应连续发言两次。
    # 只能从以下参与者中选择：
    # - {REVIEWER_NAME}
    # - {FRONTDESK_NAME}
    # 始终遵循以下规则选择下一位参与者，每次对话至少4轮：
    # - 用户输入后，轮到{FRONTDESK_NAME}
    # - {FRONTDESK_NAME}回复后，轮到{REVIEWER_NAME}
    # - {REVIEWER_NAME}提供反馈后，轮到{FRONTDESK_NAME}
    # 历史记录：
    # {{$history}}
    selection_function = KernelFunctionFromPrompt(
        function_name="selection",
        prompt=f"""
        Determine which participant takes the next turn in a conversation based on the the most recent participant.
        State only the name of the participant to take the next turn.
        No participant should take more than one turn in a row.
        
        Choose only from these participants:
        - {REVIEWER_NAME}
        - {FRONTDESK_NAME}
        
        Always follow these rules when selecting the next participant, each conversation should be at least 4 turns:
        - After user input, it is {FRONTDESK_NAME}'s turn.
        - After {FRONTDESK_NAME} replies, it is {REVIEWER_NAME}'s turn.
        - After {REVIEWER_NAME} provides feedback, it is {FRONTDESK_NAME}'s turn.

        History:
        {{$history}}
        """,
    )

    # 创建多Agent对话组
    chat = AgentGroupChat(
        agents=[agent_writer, agent_reviewer],  # 注册所有参与Agent
        termination_strategy=KernelFunctionTerminationStrategy(
            agents=[agent_reviewer],  # 指定由礼宾Agent判断是否终止
            function=termination_function,
            kernel=_create_kernel_with_chat_completion(),
            # 解析函数：将"是"解释为对话完成
            result_parser=lambda result: str(result.value[0]).lower() == "yes",
            history_variable_name="history",
            maximum_iterations=10,  # 最大对话轮次
        ),
        selection_strategy=KernelFunctionSelectionStrategy(
            function=selection_function,
            kernel=_create_kernel_with_chat_completion(),
            # 解析函数：将结果转换为Agent名称
            result_parser=lambda result: str(
                result.value[0]) if result.value is not None else FRONTDESK_NAME,
            agent_variable_name="agents",
            history_variable_name="history",  # 与提示中的变量名一致
        ),
    )

    user_input = "I would like to go to Paris."

    # 添加用户消息到对话历史
    await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=user_input))
    print(f"# User: '{user_input}'")

    # 启动对话流程，自动在Agents之间切换
    async for content in chat.invoke():
        # 显示Agent名称和内容
        print(f"# Agent - {content.name or '*'}: '{content.content}'")

    # 显示对话是否完成
    print(f"# IS COMPLETE: {chat.is_complete}")


await main()

Function failed. Error: Argument 'history' has a value that doesn't support automatic encoding. Set allow_dangerously_set_content to 'True' for this argument and implement custom encoding, or provide the value as a string.
Kernel Function Selection Strategy next method failed
Traceback (most recent call last):
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\semantic_kernel\agents\strategies\selection\kernel_function_selection_strategy.py", line 95, in select_agent
    result = await self.function.invoke(kernel=self.kernel, arguments=arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 290, in invoke
    raise e
  File "c:\Users\bangsun\miniconda3\envs\agentlearn\Lib\site-packages\semantic_kernel\functions\kernel_function.py", line 275, in invoke
    await stack(function_context)
  File "c:\Users\bangsun\mi

# User: 'I would like to go to Paris.'


AgentChatException: Failed to select agent


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
